# Feature Engineering
Building model-ready features from cleaned Strava data.
Features: rolling load, ACWR, HR efficiency, temporal features, and Whoop biometrics.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv("../data/processed/activities_clean.csv", parse_dates=["date"])
df["date"] = df["date"].dt.tz_localize(None)  # strip timezone
df = df.sort_values("date").reset_index(drop=True)

print(f"Loaded {len(df)} runs")
df.head()

Loaded 509 runs


,id,date,name,distance_km,duration_sec,elevation_m,avg_hr,max_hr,avg_speed,pace_min_per_km,week,day_of_week,month
0,5159424797,2021-04-20 12:50:09,Lunch Run,3.0163,925,0.0,NaN,NaN,3.261,5.111118,2021-04-19,Tuesday,April
1,5207562011,2021-04-28 16:18:00,Afternoon Run,2.0068,594,0.0,NaN,NaN,3.378,4.933227,2021-04-26,Wednesday,April
2,6227335431,2021-11-08 00:05:57,Midnight run,6.4928,2041,32.2,NaN,NaN,3.181,5.239137,2021-11-08,Monday,November
3,6944184769,2022-04-07 12:31:22,Lunch Run,2.5614,706,0.0,NaN,NaN,3.628,4.593842,2022-04-04,Thursday,April
4,7022950636,2022-04-22 16:16:56,Afternoon Run,2.0087,478,0.0,NaN,NaN,4.202,3.966081,2022-04-18,Friday,April


In [4]:
df["day_of_week"] = df["date"].dt.dayofweek      # 0=Monday, 6=Sunday
df["month"] = df["date"].dt.month
df["hour"] = df["date"].dt.hour
df["time_of_day"] = pd.cut(df["hour"],
                            bins=[-1, 6, 12, 17, 21, 24],
                            labels=["night", "morning", "afternoon", "evening", "late"])

print(df[["date", "hour", "time_of_day"]].head(10))

                 date  hour time_of_day
0 2021-04-20 12:50:09    12     morning
1 2021-04-28 16:18:00    16   afternoon
2 2021-11-08 00:05:57     0       night
3 2022-04-07 12:31:22    12     morning
4 2022-04-22 16:16:56    16   afternoon
5 2022-05-17 20:16:09    20     evening
6 2022-07-31 20:09:54    20     evening
7 2022-11-12 18:54:26    18     evening
8 2022-11-29 15:27:07    15   afternoon
9 2023-02-28 19:05:09    19     evening


In [3]:
df["days_since_last_run"] = df["date"].diff().dt.days.fillna(0)

print(df[["date", "days_since_last_run"]].head(10))

                 date  days_since_last_run
0 2021-04-20 12:50:09                  0.0
1 2021-04-28 16:18:00                  8.0
2 2021-11-08 00:05:57                193.0
3 2022-04-07 12:31:22                150.0
4 2022-04-22 16:16:56                 15.0
5 2022-05-17 20:16:09                 25.0
6 2022-07-31 20:09:54                 74.0
7 2022-11-12 18:54:26                103.0
8 2022-11-29 15:27:07                 16.0
9 2023-02-28 19:05:09                 91.0


In [5]:
# Set date as index for rolling calculations
df = df.set_index("date")

df["rolling_7d_km"] = df["distance_km"].rolling("7D").sum()
df["rolling_28d_km"] = df["distance_km"].rolling("28D").sum()
df["rolling_42d_km"] = df["distance_km"].rolling("42D").sum()

df = df.reset_index()

print(df[["date", "distance_km", "rolling_7d_km", "rolling_28d_km"]].tail(10))

                   date  distance_km  rolling_7d_km  rolling_28d_km
499 2026-03-22 09:03:13      26.2052        59.8519         97.4534
500 2026-03-26 18:23:14       7.5249        33.7301         93.2687
501 2026-03-29 10:59:51      21.4345        28.9594        114.7032
502 2026-04-04 10:00:08      31.3833        52.8178        146.0865
503 2026-04-08 20:02:32       6.3605        37.7438        126.5551
504 2026-04-12 09:02:42      10.6095        16.9700        137.1646
505 2026-04-14 18:48:18       8.7112        25.6812        112.2291
506 2026-04-14 20:31:01       3.5041        29.1853        115.7332
507 2026-04-19 09:10:20      42.5428        54.7581        132.0708
508 2026-05-05 16:55:11      11.0201        11.0201         82.7482


In [8]:
# Only calculate ACWR where chronic load is meaningful (>5km)
df_daily["acwr"] = np.where(
    df_daily["rolling_28d_km"] > 5,
    df_daily["rolling_7d_km"] / df_daily["rolling_28d_km"],
    np.nan
)

# Smooth it slightly
df_daily["acwr_smooth"] = df_daily["acwr"].rolling(7).mean()

# Only plot from mid 2023 when training became consistent
df_plot = df_daily[df_daily.index >= "2023-02-01"]

fig = px.line(df_plot, x=df_plot.index, y="acwr_smooth",
              title="Acute:Chronic Workload Ratio (from consistent training period)",
              labels={"acwr_smooth": "ACWR", "x": "Date"})
fig.add_hline(y=1.5, line_dash="dash", line_color="red",
              annotation_text="Injury risk threshold")
fig.add_hline(y=0.8, line_dash="dash", line_color="orange",
              annotation_text="Undertraining threshold")
fig.show()

In [23]:
# Filter to runs with valid HR data
df_hr = df[df["avg_hr"].notna()].copy()

# HR efficiency = speed (m/min) / heart rate
# Higher = more speed per heartbeat = better fitness
df_hr["speed_m_per_min"] = (df_hr["distance_km"] * 1000) / (df_hr["duration_sec"] / 60)
df_hr["hr_efficiency"] = df_hr["speed_m_per_min"] / df_hr["avg_hr"]

# Filter to easy runs only (avg_hr < 155) to remove intensity confounding
df_easy = df_hr[df_hr["avg_hr"] < 190].copy()
df_easy = df_easy[df_easy["hr_efficiency"] < 1.78].copy()
print(f"Runs with HR: {len(df_easy)}")

fig = px.scatter(df_easy, x="date", y="hr_efficiency",
                 trendline="lowess",
                 title="HR Efficiency Over Time",
                 labels={"hr_efficiency": "HR Efficiency (m/min/bpm)", "date": "Date"})
fig.show()

Runs with HR: 307


In [25]:
# Remove outliers — anything above 1.8 is likely a sensor error
df_easy = df_easy[df_easy["hr_efficiency"] < 1.78].copy()
print(f"Clean easy runs: {len(df_easy)}")

Clean easy runs: 307


In [7]:
df_easy["month_year"] = df_easy["date"].dt.to_period("M").dt.start_time

monthly_eff = df_easy.groupby("month_year")["hr_efficiency"].mean().reset_index()

fig = px.line(monthly_eff, x="month_year", y="hr_efficiency",
              title="Monthly Average HR Efficiency",
              labels={"hr_efficiency": "HR Efficiency (m/min/bpm)", 
                      "month_year": "Month"})
fig.show()

In [12]:
# Merge rolling load features from df_daily back to df
df_daily_features = df_daily[["rolling_7d_km", "rolling_28d_km", 
                               "rolling_42d_km", "acwr"]].reset_index()
df_daily_features.columns = ["date", "rolling_7d_km", "rolling_28d_km", 
                              "rolling_42d_km", "acwr"]

# Round date to day for merging
df["date_day"] = df["date"].dt.floor("D")
df_daily_features["date"] = pd.to_datetime(df_daily_features["date"])

df = df.merge(df_daily_features, left_on="date_day", right_on="date", 
              how="left", suffixes=("", "_daily"))
df = df.drop(columns=["date_daily", "date_day"])

print(df.columns.tolist())
print(df.shape)

['date', 'id', 'name', 'distance_km', 'duration_sec', 'elevation_m', 'avg_hr', 'max_hr', 'avg_speed', 'pace_min_per_km', 'week', 'day_of_week', 'month', 'hour', 'time_of_day', 'days_since_last_run', 'rolling_7d_km', 'rolling_28d_km', 'rolling_42d_km', 'acwr', 'rolling_7d_km_daily', 'rolling_28d_km_daily', 'rolling_42d_km_daily', 'acwr_daily']
(509, 24)


In [13]:
df.to_csv("../data/processed/activities_features.csv", index=False)
print("Saved feature engineered data")

Saved feature engineered data


In [17]:
recovery = pd.read_csv("../data/raw/whoop_recovery.csv", parse_dates=["date"])
sleep = pd.read_csv("../data/raw/whoop_sleep.csv", parse_dates=["date"])
cycles = pd.read_csv("../data/raw/whoop_cycles.csv", parse_dates=["date"])

print(f"Recovery: {len(recovery)} records")
print(f"Sleep: {len(sleep)} records")
print(f"Cycles: {len(cycles)} records")

recovery.head()

Recovery: 462 records
Sleep: 498 records
Cycles: 476 records


,date,cycle_id,recovery_score,hrv_rmssd,resting_hr,skin_temp_celsius,spo2_pct
0,2026-06-06,1549755130,92.0,36.027405,59.0,34.713670,93.40
1,2026-06-05,1547272575,66.0,27.062971,59.0,34.653000,96.00
2,2026-06-04,1544968588,42.0,20.632162,68.0,34.386898,93.80
3,2026-06-03,1542656715,21.0,50.097380,68.0,35.216167,95.25
4,2026-06-02,1540383934,48.0,35.557350,66.0,34.834835,95.00


In [16]:
recovery = pd.read_csv("../data/raw/whoop_recovery.csv")
print(recovery.head())
print(recovery.isnull().sum())

         date    cycle_id  recovery_score  hrv_rmssd  resting_hr  \
0  2026-06-06  1549755130            92.0  36.027405        59.0   
1  2026-06-05  1547272575            66.0  27.062971        59.0   
2  2026-06-04  1544968588            42.0  20.632162        68.0   
3  2026-06-03  1542656715            21.0  50.097380        68.0   
4  2026-06-02  1540383934            48.0  35.557350        66.0   

   skin_temp_celsius  spo2_pct  
0          34.713670     93.40  
1          34.653000     96.00  
2          34.386898     93.80  
3          35.216167     95.25  
4          34.834835     95.00  
date                 0
cycle_id             0
recovery_score       0
hrv_rmssd            0
resting_hr           0
skin_temp_celsius    2
spo2_pct             0
dtype: int64


In [18]:
# Merge recovery and cycles on date
whoop = recovery.merge(cycles, on="date", how="outer", suffixes=("_rec", "_cyc"))
whoop = whoop.merge(sleep, on="date", how="outer")
whoop = whoop.sort_values("date").reset_index(drop=True)

print(f"Whoop daily records: {len(whoop)}")
print(whoop.columns.tolist())
whoop.head()

Whoop daily records: 670
['date', 'cycle_id', 'recovery_score', 'hrv_rmssd', 'resting_hr', 'skin_temp_celsius', 'spo2_pct', 'strain', 'avg_hr', 'max_hr', 'kilojoules', 'sleep_performance_pct', 'sleep_duration_min', 'deep_sleep_min', 'rem_sleep_min', 'light_sleep_min']


,date,cycle_id,recovery_score,hrv_rmssd,resting_hr,skin_temp_celsius,spo2_pct,strain,avg_hr,max_hr,kilojoules,sleep_performance_pct,sleep_duration_min,deep_sleep_min,rem_sleep_min,light_sleep_min
0,2024-11-19,NaN,NaN,NaN,NaN,NaN,NaN,5.819178,79.0,139.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-21,721635486.0,24.0,26.576075,70.0,35.1,96.12500,6.646106,74.0,143.0,NaN,24.0,0.0,17.304050,26.436217,86.021517
2,2024-11-21,721635486.0,24.0,26.576075,70.0,35.1,96.12500,6.646106,74.0,143.0,NaN,64.0,0.0,67.773717,94.691117,157.640950
3,2024-11-22,722549675.0,30.0,22.197601,71.0,34.8,94.09091,13.985368,68.0,181.0,NaN,100.0,0.0,79.309200,57.199367,295.589100
4,2024-11-22,722549675.0,30.0,22.197601,71.0,34.8,94.09091,4.394290,70.0,132.0,NaN,100.0,0.0,79.309200,57.199367,295.589100


In [19]:
fig = px.line(whoop, x="date", y="hrv_rmssd",
              title="HRV Over Time (WHOOP)",
              labels={"hrv_rmssd": "HRV rMSSD (ms)", "date": "Date"})
fig.show()

In [20]:
fig = px.line(whoop, x="date", y="recovery_score",
              title="Daily Recovery Score (WHOOP)",
              labels={"recovery_score": "Recovery Score (%)", "date": "Date"})
fig.add_hline(y=67, line_dash="dash", line_color="green",
              annotation_text="Green zone")
fig.add_hline(y=33, line_dash="dash", line_color="red",
              annotation_text="Red zone")
fig.show()

In [21]:
# Load activities with features
df = pd.read_csv("../data/processed/activities_features.csv", parse_dates=["date"])
df["date"] = df["date"].dt.tz_localize(None)
df["date_day"] = df["date"].dt.floor("D")

# Load Whoop
recovery = pd.read_csv("../data/raw/whoop_recovery.csv", parse_dates=["date"])
sleep = pd.read_csv("../data/raw/whoop_sleep.csv", parse_dates=["date"])

# Whoop recovery from previous night affects today's run
# So shift date forward by 1 day
recovery["run_date"] = recovery["date"] + pd.Timedelta(days=1)
sleep["run_date"] = sleep["date"] + pd.Timedelta(days=1)

# Merge
df = df.merge(recovery.drop(columns=["date", "cycle_id"]), 
              left_on="date_day", right_on="run_date", how="left")
df = df.merge(sleep.drop(columns=["date"]), 
              left_on="date_day", right_on="run_date", how="left",
              suffixes=("", "_sleep"))

df = df.drop(columns=["run_date", "run_date_sleep"], errors="ignore")

print(f"Final dataset: {df.shape}")
print(f"Runs with Whoop data: {df['recovery_score'].notna().sum()}")

Final dataset: (597, 35)
Runs with Whoop data: 323


In [22]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated(subset=['id']).sum()}")
print(f"Unique activity IDs: {df['id'].nunique()}")

Duplicate rows: 88
Unique activity IDs: 509


In [23]:
# Keep only the longest sleep per day (main sleep, not naps)
sleep_main = sleep.copy()
sleep_main["run_date"] = sleep_main["date"] + pd.Timedelta(days=1)
sleep_main = sleep_main.sort_values("sleep_duration_min", ascending=False)
sleep_main = sleep_main.drop_duplicates(subset=["run_date"], keep="first")

# Redo the merge cleanly
df = pd.read_csv("../data/processed/activities_features.csv", parse_dates=["date"])
df["date"] = df["date"].dt.tz_localize(None)
df["date_day"] = df["date"].dt.floor("D")

recovery["run_date"] = recovery["date"] + pd.Timedelta(days=1)

df = df.merge(recovery.drop(columns=["date", "cycle_id"]),
              left_on="date_day", right_on="run_date", how="left")
df = df.merge(sleep_main.drop(columns=["date"]),
              left_on="date_day", right_on="run_date", how="left",
              suffixes=("", "_sleep"))

df = df.drop(columns=["run_date", "run_date_sleep"], errors="ignore")

print(f"Final dataset: {df.shape}")
print(f"Duplicate check: {df.duplicated(subset=['id']).sum()}")
print(f"Runs with Whoop data: {df['recovery_score'].notna().sum()}")

Final dataset: (519, 35)
Duplicate check: 10
Runs with Whoop data: 245


In [24]:
# Deduplicate recovery as well — keep highest recovery score per day
recovery_main = recovery.copy()
recovery_main["run_date"] = recovery_main["date"] + pd.Timedelta(days=1)
recovery_main = recovery_main.sort_values("recovery_score", ascending=False)
recovery_main = recovery_main.drop_duplicates(subset=["run_date"], keep="first")

# Deduplicate sleep — keep longest sleep per day
sleep_main = sleep.copy()
sleep_main["run_date"] = sleep_main["date"] + pd.Timedelta(days=1)
sleep_main = sleep_main.sort_values("sleep_duration_min", ascending=False)
sleep_main = sleep_main.drop_duplicates(subset=["run_date"], keep="first")

# Redo merge cleanly
df = pd.read_csv("../data/processed/activities_features.csv", parse_dates=["date"])
df["date"] = df["date"].dt.tz_localize(None)
df["date_day"] = df["date"].dt.floor("D")

df = df.merge(recovery_main.drop(columns=["date", "cycle_id"]),
              left_on="date_day", right_on="run_date", how="left")
df = df.merge(sleep_main.drop(columns=["date"]),
              left_on="date_day", right_on="run_date", how="left",
              suffixes=("", "_sleep"))

df = df.drop(columns=["run_date", "run_date_sleep"], errors="ignore")

# Final dedup safety net
df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

print(f"Final dataset: {df.shape}")
print(f"Duplicate check: {df.duplicated(subset=['id']).sum()}")
print(f"Runs with Whoop data: {df['recovery_score'].notna().sum()}")

Final dataset: (509, 35)
Duplicate check: 0
Runs with Whoop data: 235


In [25]:
df.to_csv("../data/processed/activities_final.csv", index=False)
print("Saved final dataset — 509 runs, 35 features")
print(df.columns.tolist())

Saved final dataset — 509 runs, 35 features
['date', 'id', 'name', 'distance_km', 'duration_sec', 'elevation_m', 'avg_hr', 'max_hr', 'avg_speed', 'pace_min_per_km', 'week', 'day_of_week', 'month', 'hour', 'time_of_day', 'days_since_last_run', 'rolling_7d_km', 'rolling_28d_km', 'rolling_42d_km', 'acwr', 'rolling_7d_km_daily', 'rolling_28d_km_daily', 'rolling_42d_km_daily', 'acwr_daily', 'date_day', 'recovery_score', 'hrv_rmssd', 'resting_hr', 'skin_temp_celsius', 'spo2_pct', 'sleep_performance_pct', 'sleep_duration_min', 'deep_sleep_min', 'rem_sleep_min', 'light_sleep_min']
